In [1]:
# Run once if dependencies are missing, then restart the kernel:
# %pip install torch datasets
import math
import random
from dataclasses import dataclass
import torch
from torch import nn
import torch.nn.functional as F
from datasets import load_dataset

SEED = 42
random.seed(SEED)
torch.manual_seed(SEED)
torch.set_num_threads(min(4, torch.get_num_threads()))
DEVICE = torch.device("cpu")  # CPU is sufficient for these small correctness tests.
print("PyTorch:", torch.__version__, "| device:", DEVICE)

PyTorch: 2.14.0 | device: cpu


/opt/miniconda3/envs/vscode/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
DATASET_ID = "HuggingFaceFW/fineweb-edu"
DATASET_CONFIG = "CC-MAIN-2024-10"
N_DOCUMENTS = 48
MAX_CHARS_PER_DOCUMENT = 3000

fw = load_dataset(DATASET_ID, name=DATASET_CONFIG, split="train", streaming=True)
texts = []
for row in fw.shuffle(seed=SEED, buffer_size=64):
    text = row.get("text", "").strip()
    if len(text) >= 100:
        texts.append(text[:MAX_CHARS_PER_DOCUMENT])
    if len(texts) == N_DOCUMENTS:
        break
if len(texts) < 8:
    raise RuntimeError("Need at least 8 nonempty documents for a train/validation split.")
cut = int(0.8 * len(texts))
train_texts, val_texts = texts[:cut], texts[cut:]
print(f"Dataset: {DATASET_ID}/{DATASET_CONFIG}")
print(f"Documents: {len(train_texts)} train, {len(val_texts)} validation")

Dataset: HuggingFaceFW/fineweb-edu/CC-MAIN-2024-10
Documents: 38 train, 10 validation


In [3]:
class CharTokenizer:
    """Temporary integration tokenizer; the team can replace it later."""
    pad_id, bos_id, eos_id, unk_id = range(4)

    def __init__(self, training_texts):
        self.chars = sorted(set("".join(training_texts)))
        self.stoi = {ch: i + 4 for i, ch in enumerate(self.chars)}
        self.itos = {i: ch for ch, i in self.stoi.items()}

    @property
    def vocab_size(self):
        return len(self.chars) + 4

    def encode(self, text):
        return [self.stoi.get(ch, self.unk_id) for ch in text]

    def decode(self, ids):
        return "".join(self.itos.get(int(i), "�" if int(i) == self.unk_id else "") for i in ids)

tokenizer = CharTokenizer(train_texts)
BLOCK_SIZE = 64

def make_windows(documents):
    windows = []
    for document in documents:
        ids = [tokenizer.bos_id] + tokenizer.encode(document) + [tokenizer.eos_id]
        # Each target is shifted by exactly one token; windows never cross documents.
        for start in range(0, len(ids) - 1, BLOCK_SIZE):
            part = ids[start:start + BLOCK_SIZE + 1]
            if len(part) >= 2:
                windows.append(part)
    return windows

train_windows, val_windows = make_windows(train_texts), make_windows(val_texts)

def get_batch(windows, batch_size=8, generator=None):
    indices = torch.randint(len(windows), (batch_size,), generator=generator).tolist()
    x = torch.full((batch_size, BLOCK_SIZE), tokenizer.pad_id, dtype=torch.long)
    y = torch.full_like(x, -100)
    for row, index in enumerate(indices):
        part = windows[index]
        length = len(part) - 1
        x[row, :length] = torch.tensor(part[:-1])
        y[row, :length] = torch.tensor(part[1:])
    return x.to(DEVICE), y.to(DEVICE)

print("Vocabulary:", tokenizer.vocab_size, "| windows:", len(train_windows), len(val_windows))
assert tokenizer.decode(tokenizer.encode("hello")) == "hello"

Vocabulary: 97 | windows: 1380 420


In [4]:
@dataclass
class GPTConfig:
    vocab_size: int
    block_size: int = 64
    n_layer: int = 2
    n_head: int = 4
    n_embd: int = 64
    dropout: float = 0.1
    pad_id: int = 0

    def __post_init__(self):
        if min(self.vocab_size, self.block_size, self.n_layer, self.n_head, self.n_embd) < 1:
            raise ValueError("Model dimensions must be positive.")
        if self.n_embd % self.n_head:
            raise ValueError("n_embd must be divisible by n_head.")
        if not 0 <= self.dropout < 1 or not 0 <= self.pad_id < self.vocab_size:
            raise ValueError("Invalid dropout or pad_id.")

class MultiHeadAttention(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.n_head = config.n_head
        self.head_dim = config.n_embd // config.n_head
        self.q_proj = nn.Linear(config.n_embd, config.n_embd)
        self.k_proj = nn.Linear(config.n_embd, config.n_embd)
        self.v_proj = nn.Linear(config.n_embd, config.n_embd)
        self.out_proj = nn.Linear(config.n_embd, config.n_embd)
        self.attn_dropout = nn.Dropout(config.dropout)
        self.resid_dropout = nn.Dropout(config.dropout)

    def forward(self, x, context=None, key_valid=None, causal=False):
        context = x if context is None else context
        batch, query_len, width = x.shape
        key_len = context.size(1)
        if causal and query_len != key_len:
            raise ValueError("Causal attention requires matching query/key lengths; no KV cache here.")
        def heads(t):
            return t.reshape(batch, -1, self.n_head, self.head_dim).transpose(1, 2)
        q, k, v = heads(self.q_proj(x)), heads(self.k_proj(context)), heads(self.v_proj(context))
        scores = (q @ k.transpose(-2, -1)) / math.sqrt(self.head_dim)
        allowed = torch.ones((batch, 1, query_len, key_len), dtype=torch.bool, device=x.device)
        if key_valid is not None:
            if key_valid.shape != (batch, key_len) or key_valid.dtype != torch.bool:
                raise ValueError("key_valid must be a bool tensor [batch, key_length].")
            allowed = allowed & key_valid[:, None, None, :]
        if causal:
            allowed = allowed & torch.ones(query_len, key_len, dtype=torch.bool, device=x.device).tril()
        weights = scores.masked_fill(~allowed, torch.finfo(scores.dtype).min).softmax(dim=-1)
        weights = weights.masked_fill(~allowed, 0.0)
        output = self.attn_dropout(weights) @ v
        output = output.transpose(1, 2).contiguous().reshape(batch, query_len, width)
        return self.resid_dropout(self.out_proj(output))

class MLP(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(config.n_embd, 4 * config.n_embd),
                                 nn.GELU(), nn.Linear(4 * config.n_embd, config.n_embd),
                                 nn.Dropout(config.dropout))
    def forward(self, x):
        return self.net(x)

class TBlock(nn.Module):
    def __init__(self, config, cross_attention=False):
        super().__init__()
        self.ln_1 = nn.LayerNorm(config.n_embd)
        self.attn = MultiHeadAttention(config)
        self.ln_2 = nn.LayerNorm(config.n_embd)
        self.mlp = MLP(config)
        self.cross_attention = cross_attention
        if cross_attention:
            self.ln_cross = nn.LayerNorm(config.n_embd)
            self.cross_attn = MultiHeadAttention(config)

    def forward(self, x, valid=None, causal=True, memory=None, memory_valid=None):
        x = x + self.attn(self.ln_1(x), key_valid=valid, causal=causal)
        if self.cross_attention:
            if memory is None:
                raise ValueError("Encoder memory is required for this decoder block.")
            x = x + self.cross_attn(self.ln_cross(x), context=memory, key_valid=memory_valid)
        x = x + self.mlp(self.ln_2(x))
        return x if valid is None else x.masked_fill(~valid.unsqueeze(-1), 0.0)

class TransformerBackbone(nn.Module):
    """Embedding-independent stack for integration into the team's model."""
    def __init__(self, config, causal=True, cross_attention=False):
        super().__init__()
        self.causal = causal
        self.blocks = nn.ModuleList([TBlock(config, cross_attention) for _ in range(config.n_layer)])
        self.ln_f = nn.LayerNorm(config.n_embd)

    def forward(self, embeddings, valid=None, memory=None, memory_valid=None):
        x = embeddings
        for block in self.blocks:
            x = block(x, valid, self.causal, memory, memory_valid)
        x = self.ln_f(x)
        return x if valid is None else x.masked_fill(~valid.unsqueeze(-1), 0.0)

In [5]:
class TokenPositionEmbedding(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.config = config
        self.token = nn.Embedding(config.vocab_size, config.n_embd, padding_idx=config.pad_id)
        self.position = nn.Embedding(config.block_size, config.n_embd)
        self.dropout = nn.Dropout(config.dropout)

    def forward(self, ids):
        if ids.ndim != 2 or ids.dtype != torch.long:
            raise ValueError("Token IDs must have shape [batch, length] and dtype torch.long.")
        if not 1 <= ids.size(1) <= self.config.block_size:
            raise ValueError("Sequence length must be between 1 and block_size.")
        positions = torch.arange(ids.size(1), device=ids.device)
        return self.dropout(self.token(ids) + self.position(positions))

def language_model_loss(logits, targets):
    if targets is None:
        return None
    if targets.shape != logits.shape[:2]:
        raise ValueError("Targets must match [batch, length]; shift them before calling the model.")
    if not (targets != -100).any():
        raise ValueError("A batch must contain at least one supervised target.")
    return F.cross_entropy(logits.reshape(-1, logits.size(-1)), targets.reshape(-1), ignore_index=-100)

class GPT(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.config = config
        self.embedding = TokenPositionEmbedding(config)
        self.backbone = TransformerBackbone(config, causal=True)
        self.lm_head = nn.Linear(config.n_embd, config.vocab_size, bias=False)

    def forward(self, idx, targets=None, attention_mask=None):
        valid = idx.ne(self.config.pad_id) if attention_mask is None else attention_mask
        hidden = self.backbone(self.embedding(idx), valid)
        logits = self.lm_head(hidden)
        return logits, language_model_loss(logits, targets)

    @torch.no_grad()
    def generate(self, idx, max_new_tokens=80, temperature=1.0, top_k=20, eos_id=None):
        if temperature <= 0 or (top_k is not None and top_k < 1):
            raise ValueError("temperature and top_k must be positive.")
        if idx.ndim != 2 or idx.size(0) != 1 or idx.size(1) == 0:
            raise ValueError("This simple generator accepts one nonempty, unpadded prompt.")
        if idx.eq(self.config.pad_id).any():
            raise ValueError("Generation prompts must not contain padding.")
        was_training = self.training
        self.eval()
        try:
            for _ in range(max_new_tokens):
                logits, _ = self(idx[:, -self.config.block_size:])
                scores = logits[:, -1, :] / temperature
                scores[:, self.config.pad_id] = -torch.inf
                if top_k is not None:
                    threshold = scores.topk(min(top_k, scores.size(-1))).values[:, -1:]
                    scores = scores.masked_fill(scores < threshold, -torch.inf)
                token = torch.multinomial(scores.softmax(-1), num_samples=1)
                idx = torch.cat((idx, token), dim=1)
                if eos_id is not None and token.item() == eos_id:
                    break
            return idx
        finally:
            self.train(was_training)

class EncoderDecoderTransformer(nn.Module):
    """Optional original-transformer-style path, using the same attention and blocks."""
    def __init__(self, config):
        super().__init__()
        self.config = config
        self.src_embedding = TokenPositionEmbedding(config)
        self.tgt_embedding = TokenPositionEmbedding(config)
        self.encoder = TransformerBackbone(config, causal=False)
        self.decoder = TransformerBackbone(config, causal=True, cross_attention=True)
        self.lm_head = nn.Linear(config.n_embd, config.vocab_size, bias=False)

    def forward(self, src_ids, decoder_ids, targets=None):
        src_valid = src_ids.ne(self.config.pad_id)
        tgt_valid = decoder_ids.ne(self.config.pad_id)
        memory = self.encoder(self.src_embedding(src_ids), src_valid)
        hidden = self.decoder(self.tgt_embedding(decoder_ids), tgt_valid, memory, src_valid)
        logits = self.lm_head(hidden)
        return logits, language_model_loss(logits, targets)

config = GPTConfig(vocab_size=tokenizer.vocab_size, block_size=BLOCK_SIZE)
model = GPT(config).to(DEVICE)
print(f"Randomly initialized GPT parameters: {sum(p.numel() for p in model.parameters()):,}")

Randomly initialized GPT parameters: 116,608


In [6]:
def run_correctness_tests():
    torch.manual_seed(SEED)
    cfg = GPTConfig(vocab_size=32, block_size=12, n_layer=2, n_head=2, n_embd=16, dropout=0.0)
    m = GPT(cfg).eval()
    ids = torch.tensor([[1, 4, 5, 6, 7, 8], [1, 9, 10, 11, 12, 13]])
    logits, loss = m(ids, ids)
    assert logits.shape == (2, 6, 32) and torch.isfinite(loss)
    changed = ids.clone(); changed[:, 3:] = 20
    torch.testing.assert_close(m(ids)[0][:, :3], m(changed)[0][:, :3], atol=1e-6, rtol=1e-5)
    valid = torch.tensor([[True, True, False, True, True, True]]).expand(2, -1)
    changed = ids.clone(); changed[:, 2] = 21
    torch.testing.assert_close(m(ids, attention_mask=valid)[0][:, 3:],
                               m(changed, attention_mask=valid)[0][:, 3:], atol=1e-6, rtol=1e-5)
    padded = F.pad(ids, (0, 2))
    torch.testing.assert_close(m(ids)[0], m(padded)[0][:, :6], atol=1e-6, rtol=1e-5)
    assert torch.isfinite(m(torch.zeros((2, 4), dtype=torch.long))[0]).all()
    labels = ids.clone(); labels[:, -2:] = -100
    actual = m(ids, labels)[1]
    expected = F.cross_entropy(logits[:, :-2].reshape(-1, 32), ids[:, :-2].reshape(-1))
    torch.testing.assert_close(actual, expected)

    a = MultiHeadAttention(cfg).eval()
    x = torch.randn(2, 5, cfg.n_embd)
    def heads(t):
        return t.reshape(2, 5, cfg.n_head, -1).transpose(1, 2)
    q, k, v = heads(a.q_proj(x)), heads(a.k_proj(x)), heads(a.v_proj(x))
    reference = F.scaled_dot_product_attention(q, k, v, is_causal=True)
    reference = a.out_proj(reference.transpose(1, 2).reshape(2, 5, cfg.n_embd))
    torch.testing.assert_close(a(x, causal=True), reference, atol=1e-6, rtol=1e-5)

    seq = EncoderDecoderTransformer(cfg).eval()
    src, dec = ids, ids[:, :4]
    seq_logits, seq_loss = seq(src, dec, dec)
    assert seq_logits.shape == (2, 4, 32)
    changed_dec = dec.clone(); changed_dec[:, 2:] = 22
    torch.testing.assert_close(seq(src, dec)[0][:, :2], seq(src, changed_dec)[0][:, :2], atol=1e-6, rtol=1e-5)
    torch.testing.assert_close(seq(src, dec)[0], seq(F.pad(src, (0, 2)), dec)[0], atol=1e-6, rtol=1e-5)
    assert not torch.allclose(seq(src, dec)[0], seq(src.flip(1), dec)[0])
    for candidate, objective in [(m, loss), (seq, seq_loss)]:
        objective.backward()
        assert all(p.grad is not None and torch.isfinite(p.grad).all() for p in candidate.parameters())
        assert all(p.grad.abs().sum() > 0 for name, p in candidate.named_parameters() if "q_proj.weight" in name)
    # Integration contract: external embeddings can be used without the token wrapper.
    external = torch.randn(2, 6, cfg.n_embd, requires_grad=True)
    m.backbone(external).square().mean().backward()
    assert external.grad is not None and torch.isfinite(external.grad).all()
    # Teacher-forced encoder–decoder prefix completion using a real dataset batch.
    real_x, real_y = get_batch(train_windows, batch_size=2)
    prefix = real_x[:, :16]
    decoder_ids = torch.cat((torch.full_like(real_x[:, :1], tokenizer.bos_id), real_x[:, 16:-1]), dim=1)
    seq_real = EncoderDecoderTransformer(config).to(DEVICE)
    real_logits, real_loss = seq_real(prefix, decoder_ids, real_x[:, 16:].masked_fill(real_x[:, 16:] == 0, -100))
    real_loss.backward()
    assert torch.isfinite(real_loss)
    assert all(p.grad is not None and torch.isfinite(p.grad).all() for p in seq_real.parameters())
    print("PASS: attention reference, shapes, causality, padding, loss, gradients, external embeddings, encoder–decoder")

run_correctness_tests()

PASS: attention reference, shapes, causality, padding, loss, gradients, external embeddings, encoder–decoder


In [7]:
torch.manual_seed(SEED)
overfit_config = GPTConfig(vocab_size=tokenizer.vocab_size, block_size=BLOCK_SIZE,
                          n_layer=1, n_head=4, n_embd=64, dropout=0.0)
probe = GPT(overfit_config).to(DEVICE)
probe_x, probe_y = get_batch(train_windows, batch_size=2)
probe_optimizer = torch.optim.AdamW(probe.parameters(), lr=0.004, weight_decay=0.0)
with torch.no_grad():
    before = probe(probe_x, probe_y)[1].item()
for _ in range(100):
    probe_optimizer.zero_grad(set_to_none=True)
    _, loss = probe(probe_x, probe_y)
    if not torch.isfinite(loss):
        raise RuntimeError("Non-finite overfit loss.")
    loss.backward()
    torch.nn.utils.clip_grad_norm_(probe.parameters(), 1.0)
    probe_optimizer.step()
with torch.no_grad():
    after = probe(probe_x, probe_y)[1].item()
print(f"Fixed-batch loss: {before:.4f} -> {after:.4f}")
assert after < before * 0.7, "Overfit test failed: investigate before integration."

Fixed-batch loss: 4.7817 -> 0.0020


In [8]:
@torch.no_grad()
def evaluate(candidate, windows, batches=8):
    was_training = candidate.training
    candidate.eval()
    rng = torch.Generator().manual_seed(123)
    total_nll, total_tokens = 0.0, 0
    try:
        for _ in range(batches):
            x, y = get_batch(windows, generator=rng)
            _, loss = candidate(x, y)
            count = int(y.ne(-100).sum())
            total_nll += loss.item() * count
            total_tokens += count
    finally:
        candidate.train(was_training)
    mean_loss = total_nll / total_tokens
    return {"loss": mean_loss, "character_perplexity": math.exp(min(mean_loss, 80))}

torch.manual_seed(SEED)
model = GPT(config).to(DEVICE)  # Fresh main model; separate from the overfit probe.
optimizer = torch.optim.AdamW(model.parameters(), lr=0.001, weight_decay=0.01)
initial_validation = evaluate(model, val_windows)
TRAIN_STEPS = 120
history = []
model.train()
for step in range(1, TRAIN_STEPS + 1):
    x, y = get_batch(train_windows)
    optimizer.zero_grad(set_to_none=True)
    _, loss = model(x, y)
    if not torch.isfinite(loss):
        raise RuntimeError(f"Non-finite training loss at step {step}.")
    loss.backward()
    grad_norm = torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0, error_if_nonfinite=True)
    optimizer.step()
    if step == 1 or step % 30 == 0:
        metrics = evaluate(model, val_windows)
        history.append({"step": step, "train_loss": loss.item(), **metrics})
        print(f"step {step:3d} | train {loss.item():.4f} | val {metrics['loss']:.4f} | char ppl {metrics['character_perplexity']:.2f}")
final_validation = evaluate(model, val_windows)
print("Initial validation:", initial_validation)
print("Final validation:  ", final_validation)
print("Validation improved:", final_validation["loss"] < initial_validation["loss"])

step   1 | train 4.7568 | val 4.7074 | char ppl 110.76
step  30 | train 3.1221 | val 3.0867 | char ppl 21.90
step  60 | train 2.8280 | val 2.8211 | char ppl 16.80
step  90 | train 2.7892 | val 2.7194 | char ppl 15.17
step 120 | train 2.7395 | val 2.6628 | char ppl 14.34
Initial validation: {'loss': 4.788918216935753, 'character_perplexity': 120.1712990462288}
Final validation:   {'loss': 2.6628183899896016, 'character_perplexity': 14.336638450156926}
Validation improved: True


In [9]:
# Generation is a mechanical check; 120 small training steps will not yield fluent text.
prompt = "The "
prompt_ids = torch.tensor([[tokenizer.bos_id] + tokenizer.encode(prompt)], dtype=torch.long, device=DEVICE)
torch.manual_seed(SEED)
generated = model.generate(prompt_ids, max_new_tokens=100, temperature=0.8,
                           top_k=20, eos_id=tokenizer.eos_id)
assert generated.shape[0] == 1 and generated.shape[1] > prompt_ids.shape[1]
assert generated.min() >= 0 and generated.max() < config.vocab_size
print(tokenizer.decode(generated[0].tolist()))

The ns o res werefoftrisalorardales des arain cs r are, tin ton, s athisehenama tinthifof msarathaloc rs
